# 04 - Social Network Analysis

This notebook builds a Yelp friendship graph for active New Orleans reviewers and exports user-level network features for forecasting.

Friendship links are static in the Yelp data, so the features are interpreted as social context, not causal influence. The active-reviewer threshold is checked with a sensitivity table before selecting the working graph.

## Design Choices

- Candidate active-reviewer thresholds: **2, 3, 5, 10, and 20** New Orleans reviews.
- Working threshold: **5** reviews, selected as a coverage/connectivity compromise.
- Nodes: active local reviewers under the selected threshold.
- Edges: Yelp friendship links where both users are active local reviewers.
- Features: active flag, degree, PageRank, component size, and Louvain community label where feasible.
- Betweenness centrality is omitted because it is expensive at this graph size.

In [ ]:
from pathlib import Path
import json
from collections import Counter

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INTERIM_DIR = PROJECT_ROOT / "data" / "interim" / "new_orleans"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "new_orleans"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures" / "sna"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

REVIEWS_PATH = INTERIM_DIR / "reviews.jsonl"
USERS_PATH = INTERIM_DIR / "users.jsonl"
USER_NETWORK_FEATURES_PATH = PROCESSED_DIR / "user_network_features.csv"
GRAPH_SUMMARY_PATH = PROCESSED_DIR / "social_graph_summary.json"
THRESHOLD_SENSITIVITY_PATH = PROCESSED_DIR / "active_reviewer_threshold_sensitivity.csv"

THRESHOLD_CANDIDATES = [2, 3, 5, 10, 20]
ACTIVE_REVIEW_THRESHOLD = 5

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})
print(PROJECT_ROOT)

In [ ]:
def iter_jsonl(path):
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            try:
                yield json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path} at line {line_number}") from exc

In [ ]:
user_review_counts = Counter()
for record in iter_jsonl(REVIEWS_PATH):
    user_review_counts[record["user_id"]] += 1

all_reviewing_user_ids = set(user_review_counts)

print(f"Reviewing users: {len(all_reviewing_user_ids):,}")
print(f"Threshold candidates: {THRESHOLD_CANDIDATES}")

In [ ]:
threshold_active_ids = {
    threshold: {uid for uid, count in user_review_counts.items() if count >= threshold}
    for threshold in THRESHOLD_CANDIDATES
}

threshold_graphs = {}
active_profile_counts = {}
for threshold, active_ids in threshold_active_ids.items():
    graph = nx.Graph()
    graph.add_nodes_from(active_ids)
    threshold_graphs[threshold] = graph
    active_profile_counts[threshold] = 0

matched_profiles = 0
for record in iter_jsonl(USERS_PATH):
    uid = record["user_id"]
    uid_review_count = user_review_counts.get(uid)
    if uid_review_count is None:
        continue
    matched_profiles += 1

    qualifying_thresholds = [threshold for threshold in THRESHOLD_CANDIDATES if uid_review_count >= threshold]
    if not qualifying_thresholds:
        continue
    for threshold in qualifying_thresholds:
        active_profile_counts[threshold] += 1

    friends = record.get("friends") or ""
    if friends == "None":
        continue
    for fid in friends.replace(", ", ",").split(","):
        fid = fid.strip()
        if not fid or uid >= fid:
            continue
        friend_review_count = user_review_counts.get(fid)
        if friend_review_count is None:
            continue
        max_shared_threshold = min(uid_review_count, friend_review_count)
        for threshold in THRESHOLD_CANDIDATES:
            if threshold <= max_shared_threshold:
                threshold_graphs[threshold].add_edge(uid, fid)

def graph_threshold_summary(threshold, graph):
    degree_values = [degree for _, degree in graph.degree()]
    components = sorted(nx.connected_components(graph), key=len, reverse=True)
    largest_component_size = len(components[0]) if components else 0
    isolated_active_users = sum(1 for value in degree_values if value == 0)
    return {
        "active_review_threshold": threshold,
        "reviewing_users": len(all_reviewing_user_ids),
        "matched_user_profiles": matched_profiles,
        "active_users": len(threshold_active_ids[threshold]),
        "active_user_share": len(threshold_active_ids[threshold]) / len(all_reviewing_user_ids),
        "graph_nodes": graph.number_of_nodes(),
        "graph_edges": graph.number_of_edges(),
        "connected_components": len(components),
        "largest_component_size": largest_component_size,
        "largest_component_share": largest_component_size / graph.number_of_nodes() if graph.number_of_nodes() else 0,
        "isolated_active_users": isolated_active_users,
        "isolated_active_user_share": isolated_active_users / graph.number_of_nodes() if graph.number_of_nodes() else 0,
        "median_friend_degree": pd.Series(degree_values).median() if degree_values else 0,
        "p90_friend_degree": pd.Series(degree_values).quantile(0.9) if degree_values else 0,
    }

threshold_sensitivity = pd.DataFrame([
    graph_threshold_summary(threshold, threshold_graphs[threshold])
    for threshold in THRESHOLD_CANDIDATES
])
threshold_sensitivity.to_csv(THRESHOLD_SENSITIVITY_PATH, index=False)
threshold_sensitivity

## Threshold Choice

The threshold controls the trade-off between coverage and graph quality. Lower thresholds keep many reviewers but include many isolated or weakly informative nodes. Higher thresholds produce a denser graph but discard most local reviewers.

The working value of **5 reviews** is kept because it preserves a meaningful active-reviewer population while producing a graph where the largest component contains more than half of active nodes. The saved sensitivity table makes this assumption explicit and allows later model comparisons across thresholds.

In [ ]:
active_user_ids = threshold_active_ids[ACTIVE_REVIEW_THRESHOLD]
G = threshold_graphs[ACTIVE_REVIEW_THRESHOLD].copy()
active_profiles = active_profile_counts[ACTIVE_REVIEW_THRESHOLD]

print(f"Selected active-reviewer threshold: {ACTIVE_REVIEW_THRESHOLD}")
print(f"Matched user profiles: {matched_profiles:,}")
print(f"Active user profiles: {active_profiles:,}")
print(f"Graph nodes: {G.number_of_nodes():,}")
print(f"Graph edges: {G.number_of_edges():,}")

In [ ]:
degree = dict(G.degree())
components = sorted(nx.connected_components(G), key=len, reverse=True)
component_id_by_user = {}
component_size_by_user = {}
for cid, nodes in enumerate(components):
    for node in nodes:
        component_id_by_user[node] = cid
        component_size_by_user[node] = len(nodes)

pagerank = nx.pagerank(G, alpha=0.85, max_iter=100, tol=1e-6) if G.number_of_edges() else {node: 0.0 for node in G.nodes}

community_by_user = {node: -1 for node in G.nodes}
community_method = "not_computed"
if components and len(components[0]) >= 3:
    largest = G.subgraph(components[0]).copy()
    try:
        communities = nx.algorithms.community.louvain_communities(largest, seed=42)
        community_method = "louvain_largest_component"
        for community_id, nodes in enumerate(communities):
            for node in nodes:
                community_by_user[node] = community_id
    except Exception as exc:
        community_method = f"not_computed: {type(exc).__name__}: {exc}"

print(f"Connected components: {len(components):,}")
print(f"Largest component size: {len(components[0]) if components else 0:,}")
print(f"Isolated active users: {sum(1 for _, d in G.degree() if d == 0):,}")
print(f"Community method: {community_method}")

In [ ]:
rows = []
for uid in all_reviewing_user_ids:
    rows.append({
        "user_id": uid,
        "subset_review_count": user_review_counts[uid],
        "active_reviewer": int(uid in active_user_ids),
        "friend_degree": degree.get(uid, 0),
        "pagerank": pagerank.get(uid, 0.0),
        "component_id": component_id_by_user.get(uid, -1),
        "component_size": component_size_by_user.get(uid, 0),
        "community_id": community_by_user.get(uid, -1),
    })

user_network_features = pd.DataFrame(rows)
user_network_features.to_csv(USER_NETWORK_FEATURES_PATH, index=False)

summary = {
    "active_review_threshold": ACTIVE_REVIEW_THRESHOLD,
    "threshold_candidates": THRESHOLD_CANDIDATES,
    "reviewing_users": len(all_reviewing_user_ids),
    "matched_user_profiles": matched_profiles,
    "active_users": len(active_user_ids),
    "graph_nodes": G.number_of_nodes(),
    "graph_edges": G.number_of_edges(),
    "connected_components": len(components),
    "largest_component_size": len(components[0]) if components else 0,
    "isolated_active_users": sum(1 for _, d in G.degree() if d == 0),
    "community_method": community_method,
    "communities_assigned": len(set(v for v in community_by_user.values() if v >= 0)),
    "threshold_sensitivity_output": str(THRESHOLD_SENSITIVITY_PATH),
    "output": str(USER_NETWORK_FEATURES_PATH),
}
with GRAPH_SUMMARY_PATH.open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)
    file.write("\n")
summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(list(degree.values()), bins=60, color="#4C78A8", edgecolor="white")
ax.set_yscale("log")
ax.set_title("Friend Degree Distribution Among Active New Orleans Reviewers")
ax.set_xlabel("Friend degree within active reviewer graph")
ax.set_ylabel("Number of users, log scale")
fig.tight_layout()
path = FIGURES_DIR / "active_user_degree_distribution.png"
fig.savefig(path, dpi=160)
plt.show()
plt.close(fig)
print(path)

In [ ]:
component_sizes = [len(c) for c in components[:20]]
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([str(i) for i in range(len(component_sizes))], component_sizes, color="#F58518")
ax.set_title("Top Connected Components in Active Reviewer Graph")
ax.set_xlabel("Component rank")
ax.set_ylabel("Number of users")
fig.tight_layout()
path = FIGURES_DIR / "top_component_sizes.png"
fig.savefig(path, dpi=160)
plt.show()
plt.close(fig)
print(path)

## Interpretation

The threshold check makes the active-reviewer definition explicit rather than arbitrary. A threshold of 5 reviews balances local coverage and graph connectivity: lower thresholds add many isolated users, while higher thresholds produce a denser but much smaller graph.

The selected graph is still fragmented, so the resulting features should be interpreted as social exposure and embeddedness rather than causal influence. Degree and PageRank capture connectedness, component size captures embeddedness, and community labels provide coarse group structure for users in the largest component.

These user-level features are aggregated to business-month features in the next notebook.